# Fine-Tuning, zweites Beispiel — ein anderer **Kopf**: Question Answering

Im ersten Notebook hatte DistilBERT einen **Classification-Kopf** (Text → ein Label).
Jetzt setzen wir denselben Körper-Typ unter einen **QuestionAnswering-Kopf**:

> **Frage + Text → die Antwort-Stelle *im* Text** (extraktives QA).

„Extraktiv" heißt: Die Antwort wird nicht frei generiert, sondern als **Ausschnitt
aus dem gegebenen Text** markiert (Start- und End-Position). Genau das kann ein
Encoder wie DistilBERT gut — er *versteht* den Text und zeigt auf die richtige Stelle.

Gleicher Ablauf wie zuvor: laden → **vorher** (Kopf untrainiert, rät) → **Fine-Tuning**
→ **nachher** (findet die Antworten).

## 1. Modell mit QA-Kopf laden

`AutoModelForQuestionAnswering` setzt einen Kopf auf, der für **jedes Wort** zwei
Werte vorhersagt: „Beginnt hier die Antwort?" und „Endet hier die Antwort?".
Dieser Kopf ist frisch und untrainiert.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import os
DEVICE = "mps" if torch.backends.mps.is_available() else (
         "cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "models/distilbert-base-uncased"
if not os.path.isdir(MODEL_ID):
    MODEL_ID = "distilbert-base-uncased"
print("Device:", DEVICE, "| Modell:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID).to(DEVICE)
print("Modell mit QA-Kopf geladen — der Kopf ist noch untrainiert.")

Device: mps | Modell: models/distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Modell mit QA-Kopf geladen — der Kopf ist noch untrainiert.


## 2. Unsere Daten: ein Text + Fragen

Bei QA besteht ein Beispiel aus **Kontext** (der Text), **Frage** und **Antwort**
— wobei die Antwort wörtlich im Kontext vorkommen muss (Ausschnitt).

In [2]:
context = (
    "Marie Curie was a physicist and chemist. "
    "She was born in Warsaw in 1867. "
    "She won the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911. "
    "She discovered the elements polonium and radium."
)

qas = [
    ("Where was Marie Curie born?",        "Warsaw"),
    ("In which year was Marie Curie born?", "1867"),
    ("What did she win in 1903?",           "Nobel Prize in Physics"),
    ("Which elements did she discover?",    "polonium and radium"),
    ("What was Marie Curie's profession?",  "physicist and chemist"),
]
print(f"1 Kontext, {len(qas)} Frage-Antwort-Paare")

1 Kontext, 5 Frage-Antwort-Paare


## 3. Eine Antwort-Funktion + Vorher-Test

Der QA-Kopf liefert für jede Position zwei Scores (Start / Ende). Die Antwort ist
der Ausschnitt vom besten Start- bis zum besten End-Wort. Wir schauen dabei nur in
den **Kontext** (nicht in die Frage).

Untrainiert zeigt der Kopf irgendwohin — meist gibt er gleich den **ganzen Text**
zurück.

In [3]:
@torch.no_grad()
def answer(question):
    model.eval()
    enc = tokenizer(question, context, return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    seq_ids = enc.sequence_ids(0)                       # 0=Frage, 1=Kontext, None=Spezialtokens
    out = model(**{k: v.to(DEVICE) for k, v in enc.items()})

    start, end = out.start_logits[0].clone(), out.end_logits[0].clone()
    for i, sid in enumerate(seq_ids):                  # alles außer Kontext „wegdrücken"
        if sid != 1:
            start[i] -= 1e4; end[i] -= 1e4
    si, ei = int(start.argmax()), int(end.argmax())
    if ei < si: ei = si
    return context[offsets[si][0]: offsets[ei][1]]

print("VORHER (QA-Kopf untrainiert):")
for q, _ in qas:
    print(f"  Q: {q}\n  A: {answer(q)!r}")

VORHER (QA-Kopf untrainiert):
  Q: Where was Marie Curie born?
  A: '.'
  Q: In which year was Marie Curie born?
  A: '. She was born in Warsaw in 1867. She won the Nobel Prize in Physics'
  Q: What did she win in 1903?
  A: 'ra'
  Q: Which elements did she discover?
  A: '.'
  Q: What was Marie Curie's profession?
  A: '.'


## 4. Daten fürs Training vorbereiten (die „Span"-Stelle)

Anders als bei der Klassifikation braucht QA als Ziel **Token-Positionen**: Wo
(in Tokens) beginnt und endet die Antwort? Wir finden die Zeichen-Position der
Antwort im Kontext und übersetzen sie mit der **Offset-Tabelle** des Tokenizers
in Start-/End-Token. Das ist der einzige Mehraufwand gegenüber Beispiel 1.

In [4]:
questions = [q for q, _ in qas]
enc = tokenizer(questions, [context] * len(qas),
                padding=True, truncation=True,
                return_offsets_mapping=True, return_tensors="pt")
offsets = enc.pop("offset_mapping")

start_positions, end_positions = [], []
for i, (q, a) in enumerate(qas):
    c0 = context.find(a); c1 = c0 + len(a)             # Zeichen-Span der Antwort
    seq_ids = enc.sequence_ids(i)
    ts = te = 0
    for ti, (o0, o1) in enumerate(offsets[i].tolist()):
        if seq_ids[ti] != 1:                           # nur Kontext-Tokens
            continue
        if o0 <= c0 < o1: ts = ti                      # Token, in dem die Antwort beginnt
        if o0 < c1 <= o1: te = ti                      # Token, in dem sie endet
    start_positions.append(ts); end_positions.append(te)

batch = {k: v.to(DEVICE) for k, v in enc.items()}
start_positions = torch.tensor(start_positions).to(DEVICE)
end_positions = torch.tensor(end_positions).to(DEVICE)
print("Start-/End-Token je Frage:", list(zip(start_positions.tolist(), end_positions.tolist())))

Start-/End-Token je Frage: [(22, 22), (26, 26), (29, 32), (47, 51), (16, 18)]


## 5. Fine-Tuning

Derselbe kleine Loop wie in Beispiel 1 — nur dass das Ziel jetzt Start-/End-
Positionen sind. Das Modell berechnet den Loss daraus selbst.

In [5]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
for epoch in range(1, 41):
    optimizer.zero_grad()
    out = model(**batch, start_positions=start_positions, end_positions=end_positions)
    out.loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoche {epoch:2d}  Loss = {out.loss.item():.4f}")
print("\nFine-Tuning fertig.")

Epoche 10  Loss = 1.8202


Epoche 20  Loss = 0.9482


Epoche 30  Loss = 0.3161


Epoche 40  Loss = 0.0835

Fine-Tuning fertig.


## 6. Nachher: dieselben Fragen

In [6]:
print("NACHHER (fine-getunt):")
for q, gold in qas:
    pred = answer(q)
    mark = "✓" if (gold.lower() in pred.lower() or pred.lower() in gold.lower()) else "✗"
    print(f"  {mark} Q: {q}\n     A: {pred!r}")

NACHHER (fine-getunt):
  ✓ Q: Where was Marie Curie born?
     A: 'Warsaw'
  ✓ Q: In which year was Marie Curie born?
     A: '1867'
  ✓ Q: What did she win in 1903?
     A: 'Nobel Prize in Physics'
  ✓ Q: Which elements did she discover?
     A: 'polonium and radium'
  ✓ Q: What was Marie Curie's profession?
     A: 'physicist and chemist'


## 7. Was wir gesehen haben

- **Gleicher Körper-Typ, anderer Kopf:** statt „Text → Label" jetzt
  „Frage + Text → Antwort-Stelle". Der Körper (Sprachverständnis) war schon da,
  der QA-Kopf war neu.
- **Gleiches Fine-Tuning-Prinzip:** vorher ratlos → mit wenigen Beispielen kurz
  nachtrainiert → findet die Antworten.
- **Der einzige Unterschied im Code:** QA braucht als Ziel **Token-Positionen**
  (Start/Ende), die wir aus der Antwort im Text berechnen.

So wurde auch der bekannte QA-Klassiker `distilbert-base-cased-distilled-squad`
gebaut — derselbe Vorgang, nur auf **SQuAD** mit ~100 000 Frage-Antwort-Paaren
statt unseren fünf.

> Merksatz: **Körper = allgemeines Können, Kopf = konkrete Aufgabe, Fine-Tuning =
> der Nachschliff.** Tausch den Kopf, und dasselbe Modell macht eine neue Aufgabe.